In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
dataset={
    'UserID':np.arange(10001,15001),
    'Tier':np.random.choice(['Basic','Standard','Premium'],size=5000),
    'MonthsActive':np.random.randint(1,13,size=5000),
    'HoursStreamed':np.random.normal(250,75,5000),
    'Region':np.random.choice(['North America','Europe','Asia','Latin America'],size=5000)
}

df=pd.DataFrame(dataset)
df['Price']=df['Tier'].map({
    'Basic':9.99,
    'Standard':14.99,
    'Premium':19.99
})
df.to_csv('dataframe.csv',index=False)
#print(df)

#method 1
def annual_value_calc():
    l=[]
    x=0
    for i in df['Tier']:
        j=df['MonthsActive'][x]
        if i=='Basic':
            value=9.99*j
            l.append(value)
        elif i=='Standard':
            value=14.99*j
            l.append(value)
        elif i=='Premium':
            value=19.99*j
            l.append(value)
        else:
            value=0
            l.append(value)
        x+=1
    return np.array(l)

print(annual_value_calc())
df['TotalRevenue']=annual_value_calc()
df.to_csv('dataframe.csv',index=False)

np.random.seed(0)
#5% of total values=250
random_rows = np.random.choice(df.index, size=250, replace=False)
df.loc[random_rows, 'HoursStreamed'] = np.nan
median=df['HoursStreamed'].median()
df=df.fillna({'HoursStreamed':median})
df1=df.loc[(df['HoursStreamed']>450) | (df['HoursStreamed']<50)]
df1.to_csv('extremeORlessUser.csv',index=False)
df2=df.loc[(df['HoursStreamed']>450)]
df2.to_csv('extremeUser.csv',index=False)
df3=df.groupby('Region')['TotalRevenue'].sum()
df4=df.groupby('Region')['MonthsActive'].mean()
print(df3)
print(df4)
df['Engagement_tier']=''
df.loc[(df['HoursStreamed']>350),'Engagement_tier']='High Engagement'
df.loc[(df['HoursStreamed']<=350)&(df['HoursStreamed']>=150),'Engagement_tier']='Medium Engagement'
df.loc[(df['HoursStreamed']<150),'Engagement_tier']='Low Engagement'
df.to_csv('dataframe.csv',index=False)
df5=df.loc[(df['Engagement_tier']=='High Engagement') & (df['MonthsActive']<6)]
df5.to_csv('Higheng6M.csv',index=False)

import matplotlib.pyplot as plt
#from DataStrAndCleaning import df 
#from DataStrAndCleaning import df3
df3=df3.sort_values(ascending=False)
print(df3)
fig, axes=plt.subplots(3,1,figsize=(10,8))

df3.plot(kind='bar',color=['blue','green','magenta','orange'],alpha=0.6,edgecolor='black',linewidth=1.2,ax=axes[0])
axes[0].set_xlabel('Region')
axes[0].set_ylabel('Tota Revenue')
axes[0].set_title('The Revenue Story')
axes[0].grid(True,alpha=0.3,axis='y')

df.plot(kind='hist',x='HoursStreamed',bins=30,color='pink',edgecolor='black',ax=axes[1],alpha=0.6)
axes[1].set_xlabel('Hours Streamed')
axes[1].set_ylabel('Number of Users')
axes[1].set_title('The Engagement distribution')
axes[1].grid(True,alpha=0.3,axis='y')

df.plot(kind='scatter',x='HoursStreamed',y='TotalRevenue',color='lightblue',ax=axes[2])
axes[2].set_xlabel('Hours Streamed')
axes[2].set_ylabel('Total Revenue')
axes[2].set_title('The Engagement vs Revenue Link')
axes[2].grid(True,alpha=0.3)

plt.tight_layout()
plt.show()